# Evaluate final gene-set accuracy

This notebook checks two model outcomes from the `itterations.ipynb` output:

- Similarity hit rate: the percentage of repeats where the final similarity reaches or exceeds the causal-gene similarity.
- Causal-gene recovery: whether the final set contains any original causal gene, and the percentage of causal genes retained in each final set.

It reads the iteration result CSVs, the matching causal-gene CSVs, and the matching pocket-candidate CSVs from `separate_disease`.

In [47]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/miasmacbook/Desktop/KCL/6-months_project")

# Use the same output folder produced by itterations.ipynb.
ITERATION_OUTPUT_ROOTS = [
    PROJECT_ROOT / "pocket_model_iterations_4_in_total",
    # PROJECT_ROOT / "pocket_model_iterations_2_in_total",
    # PROJECT_ROOT / "pocket_model_iterations_4_in_total",
]

# Set these to limit the check, or leave as None to evaluate all available iteration result CSVs.
FOLDER_NAME = None  # example: "121_gene_related_disease"
DISEASE_ID = None   # example: "EFO_0004995"

# Outputs are written like:
#   accuracy/121_gene_related_disease/final_gene_set_accuracy_summary.csv
# If FOLDER_NAME is None, each folder gets its own subfolder under accuracy/.
ACCURACY_OUTPUT_ROOT = PROJECT_ROOT / "accuracy_4"

In [48]:
from __future__ import annotations

import re
from collections import defaultdict

import pandas as pd

ENSEMBL_RE = re.compile(r"^ENSG\d+(?:\.\d+)?$")


def normalize_gene_id(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip().split(".")[0]


def split_pipe_values(value) -> list[str]:
    if pd.isna(value) or value == "":
        return []
    return [normalize_gene_id(item) for item in str(value).split("|") if normalize_gene_id(item)]


def read_causal_genes(causal_csv: Path) -> list[str]:
    causal_df = pd.read_csv(causal_csv)
    for column in ["targetId", "target_id", "gene_id", "matrix_gene_id"]:
        if column in causal_df.columns:
            return sorted(set(causal_df[column].map(normalize_gene_id).dropna()) - {""})
    raise ValueError(f"Could not find a causal-gene column in {causal_csv}. Columns: {list(causal_df.columns)}")


def parse_final_set(final_set_text: str) -> list[tuple[str, str]]:
    pairs = []
    if pd.isna(final_set_text):
        return pairs
    for part in str(final_set_text).split(" | "):
        part = part.strip()
        if not part or "->" not in part:
            continue
        seed_gene, selected_label = part.split("->", 1)
        pairs.append((normalize_gene_id(seed_gene), selected_label.strip()))
    return pairs


def build_candidate_lookup(pocket_candidates_csv: Path) -> dict[tuple[str, str], set[str]]:
    candidates = pd.read_csv(pocket_candidates_csv)
    required = {"seed_targetId", "matrix_gene_id"}
    missing = required - set(candidates.columns)
    if missing:
        raise ValueError(f"Missing columns {missing} in {pocket_candidates_csv}")

    lookup = defaultdict(set)
    label_columns = [column for column in ["candidate_label", "matrix_gene_id", "neighbor_symbol"] if column in candidates.columns]
    for _, row in candidates.iterrows():
        seed_gene = normalize_gene_id(row["seed_targetId"])
        matrix_gene = normalize_gene_id(row["matrix_gene_id"])
        if not seed_gene or not matrix_gene:
            continue
        for column in label_columns:
            label = str(row[column]).strip() if not pd.isna(row[column]) else ""
            if label:
                lookup[(seed_gene, label)].add(matrix_gene)
                lookup[(seed_gene, normalize_gene_id(label))].add(matrix_gene)
    return lookup


def resolve_final_genes(final_set_text: str, candidate_lookup: dict[tuple[str, str], set[str]]) -> tuple[list[str], list[str]]:
    resolved_genes = []
    unresolved_labels = []
    for seed_gene, selected_label in parse_final_set(final_set_text):
        matches = set()
        for label in [selected_label, normalize_gene_id(selected_label)]:
            matches.update(candidate_lookup.get((seed_gene, label), set()))
        if not matches and ENSEMBL_RE.match(normalize_gene_id(selected_label)):
            matches.add(normalize_gene_id(selected_label))
        if matches:
            resolved_genes.extend(sorted(matches))
        else:
            unresolved_labels.append(f"{seed_gene}->{selected_label}")
    return sorted(set(resolved_genes)), unresolved_labels


def find_iteration_result_csvs() -> list[Path]:
    paths = []
    for root in ITERATION_OUTPUT_ROOTS:
        if not root.exists():
            continue
        for path in root.glob("*/*/*_100_change_iterations.csv"):
            if path.name.endswith("_summary.csv"):
                continue
            folder_name = path.parent.parent.name
            disease_id = path.parent.name
            if FOLDER_NAME and folder_name != FOLDER_NAME:
                continue
            if DISEASE_ID and disease_id != DISEASE_ID:
                continue
            paths.append(path)
    return sorted(paths)


def find_score_column(df: pd.DataFrame) -> str:
    for column in ["final_score", "final score"]:
        if column in df.columns:
            return column
    raise ValueError(f"Could not find final score column. Columns: {list(df.columns)}")


def find_final_set_column(df: pd.DataFrame) -> str:
    for column in ["final_set_of_gene", "final set of gene"]:
        if column in df.columns:
            return column
    raise ValueError(f"Could not find final gene-set column. Columns: {list(df.columns)}")

In [49]:
per_repeat_rows = []
summary_rows = []

result_csvs = find_iteration_result_csvs()
print(f"Iteration result CSVs found: {len(result_csvs)}")

for result_csv in result_csvs:
    folder_name = result_csv.parent.parent.name
    disease_id = result_csv.parent.name
    disease_dir = PROJECT_ROOT / "separate_disease" / folder_name / disease_id
    causal_csv = disease_dir / f"{disease_id}.csv"
    pocket_candidates_csv = disease_dir / f"{disease_id}_pocket_candidates.csv"

    if not causal_csv.exists():
        print(f"Skipping {folder_name}/{disease_id}: missing causal CSV {causal_csv}")
        continue
    if not pocket_candidates_csv.exists():
        print(f"Skipping {folder_name}/{disease_id}: missing pocket candidates CSV {pocket_candidates_csv}")
        continue

    causal_genes = read_causal_genes(causal_csv)
    causal_gene_set = set(causal_genes)
    candidate_lookup = build_candidate_lookup(pocket_candidates_csv)
    results_df = pd.read_csv(result_csv)
    final_set_col = find_final_set_column(results_df)
    score_col = find_score_column(results_df)

    hit_col = "hit_causal_similarity" if "hit_causal_similarity" in results_df.columns else None
    if hit_col is None:
        summary_csv = result_csv.with_name(result_csv.name.replace("_100_change_iterations.csv", "_100_change_iterations_summary.csv"))
        baseline_score = None
        if summary_csv.exists():
            summary_df = pd.read_csv(summary_csv)
            if "causal_gene_similarity" in summary_df.columns and not summary_df.empty:
                baseline_score = float(summary_df.loc[0, "causal_gene_similarity"])
        if baseline_score is None:
            raise ValueError(f"No hit_causal_similarity column and no baseline score found for {result_csv}")
        similarity_hits = pd.to_numeric(results_df[score_col], errors="coerce") >= baseline_score
    else:
        similarity_hits = results_df[hit_col].astype(bool)

    disease_repeat_rows = []
    for row_index, row in results_df.iterrows():
        final_genes, unresolved_labels = resolve_final_genes(row[final_set_col], candidate_lookup)
        final_gene_set = set(final_genes)
        recovered_causal_genes = sorted(final_gene_set & causal_gene_set)
        causal_recovery_rate = len(recovered_causal_genes) / len(causal_genes) if causal_genes else 0
        repeat_row = {
            "folder_name": folder_name,
            "disease_id": disease_id,
            "source_result_csv": result_csv.as_posix(),
            "seed": row.get("seed", row_index),
            "final_score": row[score_col],
            "similarity_reached_or_above_causal": bool(similarity_hits.iloc[row_index]),
            "n_causal_genes": len(causal_genes),
            "n_final_genes_resolved": len(final_genes),
            "n_causal_genes_in_final_set": len(recovered_causal_genes),
            "any_causal_gene_in_final_set": len(recovered_causal_genes) > 0,
            "causal_gene_recovery_rate": causal_recovery_rate,
            "causal_gene_recovery_percent": causal_recovery_rate * 100,
            "causal_genes_in_final_set": "|".join(recovered_causal_genes),
            "final_genes_resolved": "|".join(final_genes),
            "unresolved_final_set_labels": "|".join(unresolved_labels),
        }
        disease_repeat_rows.append(repeat_row)
        per_repeat_rows.append(repeat_row)

    disease_repeat_df = pd.DataFrame(disease_repeat_rows)
    summary_rows.append(
        {
            "folder_name": folder_name,
            "disease_id": disease_id,
            "n_repeats": len(disease_repeat_df),
            "n_causal_genes": len(causal_genes),
            "similarity_hit_count": int(disease_repeat_df["similarity_reached_or_above_causal"].sum()),
            "similarity_hit_percent": float(disease_repeat_df["similarity_reached_or_above_causal"].mean() * 100),
            "repeat_count_with_any_causal_gene": int(disease_repeat_df["any_causal_gene_in_final_set"].sum()),
            "accuracy_any_causal_gene_percent": float(disease_repeat_df["any_causal_gene_in_final_set"].mean() * 100),
            "mean_causal_gene_recovery_percent": float(disease_repeat_df["causal_gene_recovery_percent"].mean()),
            "median_causal_gene_recovery_percent": float(disease_repeat_df["causal_gene_recovery_percent"].median()),
            "max_causal_gene_recovery_percent": float(disease_repeat_df["causal_gene_recovery_percent"].max()),
            "unique_causal_genes_seen_in_any_final_set": "|".join(sorted(set("|".join(disease_repeat_df["causal_genes_in_final_set"]).split("|")) - {""})),
        }
    )

per_repeat_df = pd.DataFrame(per_repeat_rows)
summary_df = pd.DataFrame(summary_rows)

written_outputs = []
if per_repeat_df.empty:
    print("No repeat rows were evaluated.")
else:
    for folder_name, folder_repeat_df in per_repeat_df.groupby("folder_name", sort=True):
        folder_output_dir = ACCURACY_OUTPUT_ROOT / folder_name
        folder_output_dir.mkdir(parents=True, exist_ok=True)

        folder_summary_df = summary_df[summary_df["folder_name"] == folder_name].copy()
        per_repeat_output = folder_output_dir / "final_gene_set_accuracy_per_repeat.csv"
        summary_output = folder_output_dir / "final_gene_set_accuracy_summary.csv"

        folder_repeat_df.to_csv(per_repeat_output, index=False)
        folder_summary_df.to_csv(summary_output, index=False)
        written_outputs.append((per_repeat_output, summary_output))

    for per_repeat_output, summary_output in written_outputs:
        print(f"Per-repeat output: {per_repeat_output}")
        print(f"Summary output: {summary_output}")

summary_df

Iteration result CSVs found: 602
Per-repeat output: /Users/miasmacbook/Desktop/KCL/6-months_project/accuracy_4/102_gene_related_disease/final_gene_set_accuracy_per_repeat.csv
Summary output: /Users/miasmacbook/Desktop/KCL/6-months_project/accuracy_4/102_gene_related_disease/final_gene_set_accuracy_summary.csv
Per-repeat output: /Users/miasmacbook/Desktop/KCL/6-months_project/accuracy_4/10_gene_related_disease/final_gene_set_accuracy_per_repeat.csv
Summary output: /Users/miasmacbook/Desktop/KCL/6-months_project/accuracy_4/10_gene_related_disease/final_gene_set_accuracy_summary.csv
Per-repeat output: /Users/miasmacbook/Desktop/KCL/6-months_project/accuracy_4/111_gene_related_disease/final_gene_set_accuracy_per_repeat.csv
Summary output: /Users/miasmacbook/Desktop/KCL/6-months_project/accuracy_4/111_gene_related_disease/final_gene_set_accuracy_summary.csv
Per-repeat output: /Users/miasmacbook/Desktop/KCL/6-months_project/accuracy_4/11_gene_related_disease/final_gene_set_accuracy_per_repea

,folder_name,disease_id,n_repeats,n_causal_genes,similarity_hit_count,similarity_hit_percent,repeat_count_with_any_causal_gene,accuracy_any_causal_gene_percent,mean_causal_gene_recovery_percent,median_causal_gene_recovery_percent,max_causal_gene_recovery_percent,unique_causal_genes_seen_in_any_final_set
0,102_gene_related_disease,EFO_0011011,100,102,100,100.0,100,100.0,20.490196,20.588235,30.392157,ENSG00000025293|ENSG00000049192|ENSG0000004954...
1,10_gene_related_disease,EFO_0000095,100,10,41,41.0,99,99.0,40.100000,40.000000,70.000000,ENSG00000082898|ENSG00000115524|ENSG0000014151...
2,10_gene_related_disease,EFO_0000275,100,10,100,100.0,97,97.0,28.800000,30.000000,60.000000,ENSG00000095637|ENSG00000103024|ENSG0000011713...
3,10_gene_related_disease,EFO_0001072,100,11,100,100.0,100,100.0,38.545455,36.363636,63.636364,ENSG00000105392|ENSG00000113494|ENSG0000011381...
4,10_gene_related_disease,EFO_0002422,100,10,84,84.0,99,99.0,44.100000,40.000000,80.000000,ENSG00000095002|ENSG00000099949|ENSG0000011606...
...,...,...,...,...,...,...,...,...,...,...,...,...
597,9_gene_related_disease,MONDO_0008315,100,9,100,100.0,94,94.0,28.222222,22.222222,66.666667,ENSG00000083093|ENSG00000139618|ENSG0000014882...
598,9_gene_related_disease,MONDO_0024431,100,9,100,100.0,100,100.0,41.888889,44.444444,66.666667,ENSG00000167165|ENSG00000241119|ENSG0000024163...
599,9_gene_related_disease,MONDO_0037821,100,9,100,100.0,100,100.0,43.444444,44.444444,66.666667,ENSG00000167165|ENSG00000241119|ENSG0000024163...
600,9_gene_related_disease,OBA_1000032,100,9,100,100.0,93,93.0,24.555556,22.222222,44.444444,ENSG00000010310|ENSG00000066827|ENSG0000008034...


In [44]:
# Overall model-level accuracy across all evaluated diseases/repeats.
if per_repeat_df.empty:
    print("No repeat rows were evaluated.")
else:
    overall = pd.DataFrame([
        {
            "n_diseases": summary_df["disease_id"].nunique(),
            "n_repeats": len(per_repeat_df),
            "similarity_hit_percent": per_repeat_df["similarity_reached_or_above_causal"].mean() * 100,
            "accuracy_any_causal_gene_percent": per_repeat_df["any_causal_gene_in_final_set"].mean() * 100,
            "mean_causal_gene_recovery_percent": per_repeat_df["causal_gene_recovery_percent"].mean(),
        }
    ])
    display(overall)
    display(summary_df.sort_values(["accuracy_any_causal_gene_percent", "similarity_hit_percent"], ascending=False).head(20))

,n_diseases,n_repeats,similarity_hit_percent,accuracy_any_causal_gene_percent,mean_causal_gene_recovery_percent
0,602,60200,87.109635,91.858804,50.563935


,folder_name,disease_id,n_repeats,n_causal_genes,similarity_hit_count,similarity_hit_percent,repeat_count_with_any_causal_gene,accuracy_any_causal_gene_percent,mean_causal_gene_recovery_percent,median_causal_gene_recovery_percent,max_causal_gene_recovery_percent,unique_causal_genes_seen_in_any_final_set
0,102_gene_related_disease,EFO_0011011,100,102,100,100.0,100,100.0,33.117647,33.333333,43.137255,ENSG00000025293|ENSG00000049192|ENSG0000004954...
2,10_gene_related_disease,EFO_0000275,100,10,100,100.0,100,100.0,40.300000,40.000000,70.000000,ENSG00000095637|ENSG00000103024|ENSG0000011713...
3,10_gene_related_disease,EFO_0001072,100,11,100,100.0,100,100.0,51.636364,54.545455,81.818182,ENSG00000105392|ENSG00000113494|ENSG0000011984...
5,10_gene_related_disease,EFO_0003911,100,10,100,100.0,100,100.0,39.000000,40.000000,60.000000,ENSG00000095637|ENSG00000103024|ENSG0000011713...
6,10_gene_related_disease,EFO_0004198,100,11,100,100.0,100,100.0,47.818182,45.454545,63.636364,ENSG00000083799|ENSG00000104044|ENSG0000013961...
9,10_gene_related_disease,EFO_0006925,100,10,100,100.0,100,100.0,28.400000,30.000000,50.000000,ENSG00000026652|ENSG00000112110|ENSG0000011249...
43,10_gene_related_disease,HP_0000819,100,10,100,100.0,100,100.0,36.800000,40.000000,60.000000,ENSG00000101076|ENSG00000101752|ENSG0000010663...
44,10_gene_related_disease,HP_0001915,100,10,100,100.0,100,100.0,55.600000,50.000000,80.000000,ENSG00000008710|ENSG00000106991|ENSG0000011977...
47,10_gene_related_disease,Orphanet_91088,100,10,100,100.0,100,100.0,39.000000,40.000000,70.000000,ENSG00000084674|ENSG00000095059|ENSG0000011977...
48,111_gene_related_disease,EFO_0004527,100,114,100,100.0,100,100.0,37.280702,36.842105,46.491228,ENSG00000004939|ENSG00000007384|ENSG0000000796...
